# Hälsostudie

Utredning av en hälsostudie med 800 deltagare. 
Datasetets kolumner är: Ålder, kön, längd, vikt, systoliskt blodtryck, kolesterol, rökare, sjukdom. Det är informationen som ges av deltagarna för studien.

Syftet:
- Hur ser deltagarnas ålder, vikt, längd, blodtryck och kolesterol ut?
- Hur stor del av deltagarna har sjukdomen, och stämmer det med simuleringen?
- Inom vilket intervall ligger det genomsnittliga systoliska blodtrycket?
- Har rökare högre medelblodtryck än icke-rökare?

Analysen innehåller analys, statistik, simulering, konfidensintervall och hypotesprövning

## Inläsning av data
Läser in datan med funktion "load_data" från "src/functions.py"

In [1]:
#hämta in numpy, pandas matplotlib, scipy för hypotesprövning och np.random.seed för simuleringar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42)

from src.functions import load_data, simulate_disease, ci_normal, welch_test  #läser in fler funktioner från egen
from src.health_analyzer import HealthAnalyzer  #importerar health_analyzer från src

## Inläsning av data

In [ ]:
df = load_data("data/health_study_dataset.csv")       #läs in data med egen funktion
df.head()       #Visa första 5 raderna

Den beskrivande analysen görs med klassen "HealthAnalyzer" från "src/health_analyzer.py"

In [ ]:
analys = HealthAnalyzer(df) #skapa analysator med datasetet

## Översikt av datan

In [ ]:
df.info()   #info om dataset

In [ ]:
df.describe()       #antal, medel, standardavvikelse, min, max, 25%,50% och 75%

Lägsta vikten är 33,7 kg, och lägsta åldern är 18. Låter lite omrimligt.

## Datakontroll
Kontrollera datan innan vi börjar med analysen, leta efter saknade värden, dubletter och orimliga värden.

In [ ]:
# isna/ markerar saknade värden
print(df.isna().sum())

#dublicated markerar rader som är kopior
print("Antal dubletter:", df.duplicated().sum())

In [ ]:
# value_counts() räknar hur många gånger värde förekommer, översikt över data
print(df["sex"].value_counts())
print(df["smoker"].value_counts())
print(df["disease"].value_counts())

Orimlig data förekomm tidigare och behöver utredas

In [ ]:
df[df["weight"] < 40]

Fyra deltagare väger under 40 kg. Alla är kvinnor och längden är normal. De behålls i analysen. Väldigt få personer i analysen, vilket inte påverkar resultatet väldigt lite. Behöver inte bero på fel i datan.

## Beskrivande analys

Beräkna medelvärde, median, min och max för att beskriva deltagarna. Tabellen tas fram med metoden "describe" i klassen "HealthAnalyzer".

In [ ]:
kolumner = ["age", "weight", "height", "systolic_bp", "cholesterol"]    
analys.describe(kolumner)

Tolkning: Medelvärde och median ligger nära varandra för alla variabler, så fördelningarna är jämnt fördelade. Deltagarna är mellan 18 och 90 år. Medelblodtrycket är cirka 149 mmhg, vilket är högt eftersom över 140 mmHg räknas som förhöjt.

### Fördelning av systoliskt blodtryck
Ett histogram visar hur blodtrycket fördelar sig bland deltagarna.

In [ ]:
plt.figure(figsize=(8, 5))      #Skapa figur

plt.hist(df["systolic_bp"], bins=20)    #histogram, 

plt.axvline(df["systolic_bp"].mean(), color="red", linestyle="--", label="Medelvärde") #v6, uppgift 3

plt.title("Fördelning av systoliskt blodtryck") #titel
plt.xlabel("Systoliskt blodtryck (mmHg)") #titel för x
plt.ylabel("Antal deltagare") #titel för y
plt.legend()    #v6, uppgift 4. Gör legenden tydlig med rätt ettiketer
plt.grid(True) #rutnät
plt.show() #visa grafen



Deltagarnas blodtryck är mellan cirka 130 och 170 mmHg. Värdena är fördelade kring medelvärdet, vilket stämmer emd att meddel och median låg nära varandra.

### Vikt per kön
En boxplot jämför viktfördelningen mellan kvinnor och män.

In [ ]:
# Plocka ut vikterna för varje kön
vikt_kvinnor = df[df["sex"] == "F"]["weight"]
vikt_man = df[df["sex"] == "M"]["weight"]

plt.figure(figsize=(8, 5))

plt.boxplot([vikt_kvinnor, vikt_man], tick_labels=["Kvinnor", "Män"])   # boxplot v6, uppgift 3, uppgift 1. Namnen under lådorna

plt.title("Vikt per kön")
plt.ylabel("Vikt (kg)")
plt.grid(True)
plt.show()

Män har en högre medianvikt än kvinnor (cirka 79 kg gentemot 67 kg), medan spridningen är ungefär lika stor i båda grupperna. Hos båda grupperna finns det avvikande låga värden (kvinnor runt 34 kg och män runt 45 kg). De låga värdena hos kvinnorna är samma deltagare som i datakontrollen och boxploten bekräftar att de avviker från resten av gruppen.

### Andel rökare
Ett stapeldigram visar hur stor del av deltagarna som röker.

In [ ]:
antal_rokare = df["smoker"].value_counts()      #Räkna antal deltagare per kategori i smoker
andel_rokare = antal_rokare / len(df) * 100     # Gör om andel i procent antal/totalt antal deltagare * 100, len ger anatal rader i datasetet alltså 800. (v6, uppgift 6)

print(andel_rokare.round(1))        #Skriv ut andelarna avrundade till deimaltal

plt.bar(andel_rokare.index, andel_rokare.values) #Stapeldiagram. Index = Kategorierna (No, yes) på x axeln, Values, andelarna i procent staplarnas höjd

plt.title("Andel rökare bland deltagarna")
plt.xlabel("Rökare")
plt.ylabel("Andel deltagare (%)")   #Rubriker och underrubriker på olika axlarna

plt.grid(True) #Rutnät, lättare att läsa
plt.show()


Cirka 27 procent röker av deltagarna och 73 procent röker inte. Grupperna är olika stora (213 + 587 personer). Kommer användas i hypotesprövningen senare.

### Simulering

Simulering görs med funktionen "simulate_disease" från "src/functions.py" 

In [ ]:
p_verklig = df["disease"].mean()    #andel sjuka
p_simulerad = simulate_disease(p_verklig, 1000) #simulering 1000 personer med funktion från src

print("Verklig andel: ", round(p_verklig, 4))
print("Simulerad andel: ", round(p_simulerad, 4))
print("Skillnad: ", round(p_simulerad - p_verklig, 4))

Cirka 5,9 procent av deltagarna har sjukdomen, 47 av 800 personer.
När vi simulerar 1000 personer med samma risk blir 6,5 procent sjuka. Siffrorna är inte exakt lika eftersom simuleringen gör att resultatet varierar. 
Simuleringen stämmer bra med datan.

### Konfidensintervall för systolisk blodtryck
Ett 95 procentigt konfidensintervall visar mellan vilka värden medelblodtrycket troligtvis ligger.
Normalapproximation används. Beräkningen görs med funktionen "ci_normal" från "src/functions.py"

In [ ]:
medel, nedre, ovre = ci_normal(df["systolic_bp"]) # src funktion

print("Medelvärde:", round(medel, 2))
print("95 % CI:", round(nedre, 2), "-", round(ovre, 2))

Medelblodtrycket är 149,2 mmHg och 95 procent konfidensintervallet är 148,3-150,1 mmHg. 
Det betyder att medelblodtrycket med stor sannolikhet ligger mellan värdena.

## Hypotesprövning: Rökare har högre medelblodtryck än ickerökare
Hypotesprövningen ska utreda om påstående stämmer

In [ ]:
grupper = df.groupby("smoker")["systolic_bp"].agg(["count", "mean", "std"])     # per grupp (v6 uppgift 6)

grupper.round(2) #avrunda till 2 decimaler

### Hypoteser
Nollhypotes: Rökare har inte högre medelblodtryck än ickerökare.

Mothypotes: Rökare har högre medelblodtryck än ickerökare.

Jag testar att påståendet att "rökare har högre blodtryck" därför gör jag ett ensidigt test. Signifikansnivån är 5 procent.

Testet görs med funktion "welch_test" från "src/functions.py"

In [ ]:
rokare = df[df["smoker"] == "Yes"]["systolic_bp"]   #blodtryck rökare
icke_rokare = df[df["smoker"] == "No"]["systolic_bp"] #blodtryck icke rökare

t_stat, p_varde = welch_test(rokare, icke_rokare) # ensidigt Welch t-test

print("t-värde:", round(t_stat, 4)) #skillnad i förhållande till standardfel
print("p-värde:", round(p_varde, 4)) # jämförs med signifikansnivån 0,05

### Resultat

p-värdet blev 0,33 vilket är högre än signifikansnivån 0,05. Nollhypotesen kan inte uteslutas. Rökarna har 0,5 mmHg högre medelblodtryck än ickeröckarna vilket är en väldigt liten skillnad. Resultatet visar inte att rökare har högre blodtryck.

### Slutsats

Deltagarna är mellan 18 och 90 år. Medelblodtrycket är cirka 149 mmHg vilket räknas som förhöjt. Fyra kvinnor väger under 40 kg, men jag valde att behålla dom i analysen.

Ungefär 6 procent av deltagarna har sjukdomen. Simuleringen gav i princip samma andel och den skillnaden beror på slumpmässig variation.

Konfidensintervallet visar att det verkliga medelblodtrycket med 95 procent säkerhet ligger mellan 148,3 och 150,1 mmHg.

Det går inte att påvisa att rökarna har ett högre blodtryck än icke rökarna eftersom det var 0,47 mmHg högre.